In [0]:
storageAccountName = "datalakeeb173f90c7e0bc50"
storageAccountAccessKey = ""
sasToken = "sv=2024-11-04&ss=bfqt&srt=sco&sp=rwdlacupyx&se=2025-06-06T06:54:58Z&st=2025-06-05T22:54:58Z&spr=https&sig=rCr16f24u6jMldql1xRMuplwWvT646ItW%2FJurgEsQQ4%3D"

def mount_adls(blobContainerName):
    try:
      dbutils.fs.mount(
        source = "wasbs://{}@{}.blob.core.windows.net".format(blobContainerName, storageAccountName),
        mount_point = f"/mnt/{storageAccountName}/{blobContainerName}",
        #extra_configs = {'fs.azure.account.key.' + storageAccountName + '.blob.core.windows.net': storageAccountAccessKey}
        extra_configs = {'fs.azure.sas.' + blobContainerName + '.' + storageAccountName + '.blob.core.windows.net': sasToken}
      )
      print("OK!")
    except Exception as e:
      print("Falha", e)

In [0]:
mount_adls('gold')

In [0]:
df_assistencias  = spark.read.format('delta').load(f"/mnt/{storageAccountName}/silver/assistencias")
df_atores        = spark.read.format('delta').load(f"/mnt/{storageAccountName}/silver/atores")
df_avaliacoes    = spark.read.format('delta').load(f"/mnt/{storageAccountName}/silver/avaliacoes")
df_episodios     = spark.read.format('delta').load(f"/mnt/{storageAccountName}/silver/episodios")
df_filmes        = spark.read.format('delta').load(f"/mnt/{storageAccountName}/silver/filmes")
df_generos       = spark.read.format('delta').load(f"/mnt/{storageAccountName}/silver/generos")
df_pagamentos    = spark.read.format('delta').load(f"/mnt/{storageAccountName}/silver/pagamentos")
df_planos        = spark.read.format('delta').load(f"/mnt/{storageAccountName}/silver/planos")
df_series        = spark.read.format('delta').load(f"/mnt/{storageAccountName}/silver/series")
df_usuarios      = spark.read.format('delta').load(f"/mnt/{storageAccountName}/silver/usuarios")

In [ ]:
# Query para calcular o MRR
kpi_mrr_query = """
    SELECT
        date_trunc('MONTH', data_pagamento) AS mes_referencia,
        SUM(valor_pago) AS mrr
    FROM pagamentos
    WHERE
        status_pagamento = 'Aprovado'
    GROUP BY
        mes_referencia
    ORDER BY
        mes_referencia
"""

# Executar a query e criar o DataFrame
kpi_mrr_df = spark.sql(kpi_mrr_query)

# Exibir os resultados
print("KPI 1: Receita Recorrente Mensal (MRR)")
kpi_mrr_df.display()

# Salvar o resultado na camada Gold
kpi_mrr_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.kpi_mrr_mensal")

In [ ]:
# Query para calcular a base de usuários ativos e inativos por mês
kpi_churn_query = """
    WITH status_mensal_usuarios AS (
      SELECT
        id_usuario,
        date_trunc('MONTH', data_registro) as mes_entrada,
        -- Considera o status atual; em um cenário real, usaríamos uma tabela de histórico de status (SCD Type 2)
        status_conta
      FROM usuarios
    ),
    contagem_status AS (
      SELECT
        mes_entrada,
        SUM(CASE WHEN status_conta = 'Ativo' THEN 1 ELSE 0 END) as total_ativos_no_mes,
        SUM(CASE WHEN status_conta = 'Inativo' THEN 1 ELSE 0 END) as total_inativos_no_mes
      FROM status_mensal_usuarios
      GROUP BY mes_entrada
    )
    SELECT
      mes_entrada,
      total_inativos_no_mes,
      (total_ativos_no_mes + total_inativos_no_mes) AS base_total_usuarios,
      -- Fórmula de Churn: (Inativos / Base Total) * 100
      (total_inativos_no_mes / (total_ativos_no_mes + total_inativos_no_mes)) * 100 AS taxa_churn_percentual
    FROM contagem_status
    ORDER BY mes_entrada
"""

# Executar a query
kpi_churn_df = spark.sql(kpi_churn_query)

# Exibir os resultados
print("KPI 2: Taxa de Churn Mensal")
kpi_churn_df.display()

# Salvar na camada Gold
kpi_churn_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.kpi_taxa_churn_mensal")

In [ ]:
# Query para calcular o ARPPU mensal
kpi_arppu_query = """
    WITH receita_por_usuario_pagante AS (
        SELECT
            date_trunc('MONTH', data_pagamento) as mes_referencia,
            SUM(valor_pago) as receita_total_mensal,
            COUNT(DISTINCT id_usuario) as usuarios_pagantes_unicos
        FROM pagamentos
        WHERE status_pagamento = 'Aprovado'
        GROUP BY mes_referencia
    )
    SELECT
        mes_referencia,
        receita_total_mensal / usuarios_pagantes_unicos as arppu
    FROM receita_por_usuario_pagante
    ORDER BY mes_referencia
"""

# Executar a query
kpi_arppu_df = spark.sql(kpi_arppu_query)

# Exibir os resultados
print("KPI 3: Receita Média Mensal por Usuário Pagante (ARPPU)")
kpi_arppu_df.display()

# Salvar na camada Gold
kpi_arppu_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.kpi_arppu_mensal")

In [ ]:
# Query para analisar o engajamento por gênero
kpi_engajamento_genero_query = """
    WITH conteudo_avaliado AS (
        -- Unifica filmes e séries com suas avaliações
        SELECT a.id_avaliacao, a.nota, f.genero FROM avaliacoes a
        JOIN filmes f ON a.id_conteudo = f.id_filme AND a.tipo_conteudo = 'Filme'
        UNION ALL
        SELECT a.id_avaliacao, a.nota, s.genero FROM avaliacoes a
        JOIN series s ON a.id_conteudo = s.id_serie AND a.tipo_conteudo = 'Serie'
    )
    SELECT
        genero,
        COUNT(id_avaliacao) as numero_de_avaliacoes,
        AVG(nota) as media_de_nota
    FROM conteudo_avaliado
    GROUP BY genero
    ORDER BY media_de_nota DESC, numero_de_avaliacoes DESC
"""

# Executar a query
kpi_engajamento_genero_df = spark.sql(kpi_engajamento_genero_query)

# Exibir os resultados
print("KPI 4: Engajamento por Gênero de Conteúdo")
kpi_engajamento_genero_df.display()

# Salvar na camada Gold
kpi_engajamento_genero_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.kpi_engajamento_por_genero")

In [ ]:
# Query para contar novos usuários por mês
metrica_novos_usuarios_query = """
    SELECT
        date_trunc('MONTH', data_registro) AS mes_registro,
        COUNT(id_usuario) AS novos_usuarios
    FROM usuarios
    GROUP BY
        mes_registro
    ORDER BY
        mes_registro
"""

# Executar a query
metrica_novos_usuarios_df = spark.sql(metrica_novos_usuarios_query)

# Exibir os resultados
print("Métrica 1: Novos Usuários Registrados por Mês")
metrica_novos_usuarios_df.display()

# Salvar na camada Gold
metrica_novos_usuarios_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.metrica_novos_usuarios_mensal")

In [ ]:
# Query para contar solicitações de assistência
metrica_assistencias_query = """
    SELECT
        date_trunc('MONTH', data_solicitacao) AS mes_solicitacao,
        tipo_assistencia,
        COUNT(id_assistencia) AS total_solicitacoes
    FROM assistencias
    GROUP BY
        mes_solicitacao,
        tipo_assistencia
    ORDER BY
        mes_solicitacao,
        total_solicitacoes DESC
"""

# Executar a query
metrica_assistencias_df = spark.sql(metrica_assistencias_query)

# Exibir os resultados
print("Métrica 2: Volume de Solicitações de Assistência por Tipo e Mês")
metrica_assistencias_df.display()

# Salvar na camada Gold
metrica_assistencias_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.metrica_solicitacoes_assistencia_mensal")